# 🧪 Ablation Study: Dual-Stream Fusion WITHOUT Transformer (Binary Classification)

### **Objective:**
Evaluate the student engagement classification performance using **only the EfficientNet-B0 and OpenFace dual streams and their fusion**, feeding directly into the classification head **without temporal modeling via Transformer**.

| Component | Full Model (`colab_binary_transformer`) | This Ablation Model (`colab_binary_no_transformer`) |
|---|---|---|
| **Stream 1 (Visual CNN)** | EfficientNet-B0 (1280-dim) | EfficientNet-B0 (1280-dim) |
| **Stream 2 (Facial AUs)** | OpenFace (709-dim) | OpenFace (709-dim) |
| **Fusion Layer** | Linear Projection (1989 $\rightarrow$ 128) + LayerNorm | Linear Projection (1989 $\rightarrow$ 128) + LayerNorm |
| **Temporal Modeling** | ⚡ **3-Layer Transformer Encoder** (Self-Attention) | ❌ **Removed** (Direct Temporal Mean Pooling) |
| **Classification Head** | MLP (128 $\rightarrow$ 64 $\rightarrow$ 32 $\rightarrow$ 2) | MLP (128 $\rightarrow$ 64 $\rightarrow$ 32 $\rightarrow$ 2) |
| **Target Classes** | 2 (Engaged vs Disengaged) | 2 (Engaged vs Disengaged) |
| **Training Duration** | 60 Epochs | 60 Epochs |

---

### **Step 1: Mount Google Drive**
Click the **Play ▶** button on the left of the cell. If prompted, click **"Connect to Google Drive"** and select your Google account.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive successfully mounted!")

### **Step 2: Set Working Directory & Add to Python Path**
This locates the ablation code folder in your Google Drive and changes the current directory to it.

In [ ]:
import os
import sys

# Search common upload paths on Google Drive
possible_paths = [
    "/content/drive/MyDrive/Student_Engagement_DL/colab_binary_no_transformer",
    "/content/drive/MyDrive/colab_binary_no_transformer",
    "/content/drive/MyDrive/Binary classification/colab_binary_no_transformer",
    "/content/drive/Student_Engagement_DL/colab_binary_no_transformer",
]

working_dir = None
for p in possible_paths:
    if os.path.exists(p):
        working_dir = p
        break

if working_dir is None:
    # Fallback to standard path
    working_dir = "/content/drive/MyDrive/Student_Engagement_DL/colab_binary_no_transformer"
    os.makedirs(working_dir, exist_ok=True)

if working_dir not in sys.path:
    sys.path.insert(0, working_dir)

%cd $working_dir
print("\n📁 Current working directory:", os.getcwd())
print("📄 Available files:", os.listdir('.'))

### **Step 3: Install Required Dependencies**
Installs PyTorch, torchvision, scikit-learn, pandas, matplotlib, and seaborn.

In [ ]:
!pip install -q torch torchvision scikit-learn pandas matplotlib seaborn

### **Step 4: Verify Input Data (`sequence_fuse_features_binary`)**
Checks that your preprocessed feature files exist in Google Drive and verifies the feature dimension (`1989 = 1280 + 709`).

In [ ]:
import config_binary_fusion as config
import torch

seq_dir = config.SEQUENCE_FUSED_DIR
print(f"Checking feature folder: {seq_dir}")

if os.path.exists(seq_dir):
    files = [f for f in os.listdir(seq_dir) if f.endswith('.pt')]
    print(f"✅ Successfully found {len(files)} preprocessed feature files (.pt)!")
    if len(files) > 0:
        sample = torch.load(os.path.join(seq_dir, files[0]), map_location="cpu")
        feat_dim = sample["sequence"].shape[1]
        print(f"   - Sample video     : {files[0]}")
        print(f"   - Frame count      : {sample['sequence'].shape[0]}")
        print(f"   - Feature dimension: {feat_dim} (1280 EfficientNet-B0 + 709 OpenFace)")
        print(f"   - Class label      : {sample.get('label', 'N/A')} ({config.CLASS_NAMES[sample.get('label', 0)]})")
else:
    print(f"❌ Folder not found at {seq_dir}!")
    print("Please verify your folder name in Google Drive is 'sequence_fuse_features_binary'.")

### **Step 5: Calculate & Display Parameter Count of the Entire Architecture**
This runs the parameter counting module to inspect the parameter counts of EfficientNet-B0, OpenFace, Fusion, and the Classification Head.

In [ ]:
from models_binary_fusion import BinaryEngagementFusionModel, display_parameter_summary

model = BinaryEngagementFusionModel()
param_info = display_parameter_summary(model)

### **Step 6: Train the Ablation Model for 60 Epochs**
Trains the model using AdamW, Cosine Annealing learning rate schedule, and sliding windows. Displays per-epoch loss and accuracy and parameter count at completion.

In [ ]:
!python train_binary_fusion.py

### **Step 7: Evaluate the Ablation Model on Test Videos**
Evaluates the best checkpoint on held-out test videos using soft-voting window aggregation. Computes Accuracy, F1-Score, Precision, Recall, Confusion Matrix, and ROC Curve.

In [ ]:
!python evaluate_binary_fusion.py

### **Step 8: Visualize Evaluation Plots & Confusion Matrix**
Displays the generated Confusion Matrix, ROC Curve, and Training Curves directly in this notebook.

In [ ]:
from IPython.display import Image, display
import config_binary_fusion as config

print("📊 1. Confusion Matrix (Ablation: No Transformer):")
if os.path.exists(config.CONFUSION_MATRIX_PATH):
    display(Image(filename=config.CONFUSION_MATRIX_PATH))

print("\n📈 2. ROC Curve (Ablation: No Transformer):")
if os.path.exists(config.ROC_CURVE_PATH):
    display(Image(filename=config.ROC_CURVE_PATH))

curves_path = os.path.join(config.EVAL_OUT_DIR, "training_curves_binary_fusion_ablation.png")
print("\n📉 3. Training & Validation Curves:")
if os.path.exists(curves_path):
    display(Image(filename=curves_path))

### **Step 9: View Predictions Summary Table**
Displays the first 10 video-level predictions with probabilities.

In [ ]:
import pandas as pd
import config_binary_fusion as config

if os.path.exists(config.PREDICTIONS_CSV_PATH):
    df = pd.read_csv(config.PREDICTIONS_CSV_PATH)
    display(df.head(15))
else:
    print("Predictions file not found. Run Step 7 first.")